<img src="https://cros.ec.europa.eu/system/files/inline-images/logo%20official%20estp_0.jpg" height="120px">

# ADVANCED PYTHON FOR OFFICIAL STATISTICS
## ICON-Institute · Cologne · June 22–26, 2026 · Day 2 AM
### [Dr. Christian Kauth](https://www.linkedin.com/in/ckauth/)

---

# 🐼 Notebook 03 — Advanced Pandas & Polars
## *Advanced Python for Official Statistics*

> *"The first 80% of data cleaning takes 80% of the time. The last 20% takes the other 80%."*

Today we tackle **scale**. Luxembourg alone has 40 CSV files; all 26 countries span >10 years. We need techniques that stay fast and memory-efficient at that scale. Then we rewrite our SILC pipeline in **Polars** — a blazing-fast DataFrame library — and compare performance.

| Topic | You will be able to… |
|-------|---------------------|
| **Chunked loading** | Process multi-GB CSV files without running out of RAM |
| **Optimised dtypes** | Shrink DataFrame memory footprint |
| **Multi-index** | Query by `(country, year)` without merge or filter |
| **groupby + agg** | Compute AROP by region and year in a single expression |
| **window functions** | Rolling 3-year AROP trend per country |
| **Merging & joining** | Combine the four SILC files (D/H/R/P) with `validate` & `indicator` |
| **Reshaping** | `pivot_table`, `crosstab`, `melt` for tidy analysis tables |
| **Binning & ranking** | `cut`, `qcut`, `rank` for income deciles and age bands |
| **Survey weights** | Weighted median / quantile / AROP — the official-statistics correction |
| **Inequality** | Weighted Lorenz curve & Gini coefficient |
| **Vectorisation** | Why `.apply()` is slow and how to avoid it |
| **Polars** | Lazy evaluation, 5–10× faster than pandas on this workload |
| **Parquet** | Store 10 years of SILC data in 10% of the CSV size |

In [9]:
import os
from pathlib import Path


def find_project_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "pyproject.toml").exists():
            return candidate
    raise FileNotFoundError("pyproject.toml not found")

PROJECT_ROOT = find_project_root(Path().resolve())
os.chdir(PROJECT_ROOT)
DATA_DIR    = PROJECT_ROOT / "data"
PARQUET_DIR = DATA_DIR / "parquet"
print(f"✅ Project root: {PROJECT_ROOT}")


✅ Project root: /home/administrateur/projects/python_advanced


### `pyproject.toml` update — Notebook 03

This notebook adds: **matplotlib** (for the Lorenz-curve / inequality visualisation).

Run the cell below to update `pyproject.toml` and install the new packages.

In [10]:
%%writefile pyproject.toml
# Day 1-3: polars already included
# New in notebook 03:
#   + matplotlib (Lorenz curve / inequality visualisation)

[build-system]
requires = ["hatchling"]
build-backend = "hatchling.build"

[project]
name = "silc-toolkit"
version = "0.1.0"
description = "Advanced Python for Official Statistics -- EU-SILC Analysis Toolkit"
readme = "README.md"
requires-python = ">=3.11"
authors = [{ name = "Christian Kauth", email = "christian.kauth@unifr.ch" }]
license = { text = "MIT" }

dependencies = [
    "pydantic>=2.5",
    "pandas>=2.1",
    "pyarrow>=14.0",
    "polars>=1.40",
    "matplotlib>=3.8",
]

[project.optional-dependencies]
dev = [
    "pytest>=7.4",
    "pytest-cov>=4.1",
    "black>=23.0",
    "ruff>=0.3",
]

[project.urls]
Repository = "https://github.com/icon-institute/silc-toolkit"

[tool.hatch.build.targets.wheel]
packages = ["silc_toolkit"]

[tool.hatch.build.targets.sdist]
# Keep the source distribution lean — the PUF data is not part of the package
exclude = ["data", "dist", "src"]

[tool.pytest.ini_options]
testpaths = ["tests"]
addopts = "-v --tb=short"

[tool.black]
line-length = 88
target-version = ["py311"]

[tool.ruff]
line-length = 88
target-version = "py311"

[tool.ruff.lint]
select = ["E", "F", "I", "UP"]
ignore = ["E501"]

[tool.ruff.lint.isort]
known-first-party = ["silc_toolkit"]

Overwriting pyproject.toml


In [11]:
# Sync all dependencies declared in pyproject.toml
# Run this after every %%writefile pyproject.toml cell
import subprocess

result = subprocess.run(
    ["uv", "pip", "install", "-e", "."],
    capture_output=True, text=True
)
if result.returncode == 0:
    print("silc-toolkit + notebook 03 deps installed")
else:
    print("uv error:", result.stderr[-500:])

uv error:     ~~~~~~~^^^^^^^^^^^^^^^^^
        File
      "/home/administrateur/.cache/uv/builds-v0/.tmpA6PyII/lib/python3.13/site-packages/hatchling/metadata/core.py",
      line 533, in readme
          raise OSError(message)
      OSError: Readme file does not exist: README.md


hint: Build failures usually indicate a problem with the package or the build environment


In [4]:
import numpy as np
import pandas as pd

print(f"   pandas {pd.__version__}  |  numpy {np.__version__}")
countries = sorted(d.name[:2] for d in DATA_DIR.iterdir() if d.is_dir() and d.name.endswith("_PUF_EUSILC"))
print(f"   Countries: {', '.join(countries)}")


   pandas 3.0.3  |  numpy 2.5.0


FileNotFoundError: [Errno 2] No such file or directory: '/home/administrateur/projects/python_advanced/src/notebooks/data'

### Human-readable label maps

EU-SILC stores every categorical variable as a bare integer or terse code. We define the
**code → label** dictionaries *once, up front*, and reuse them throughout the notebook with
`Series.map(...)` plus friendly column names — so every table below reads in plain language
(`female`, `tertiary`, `Liège`, `household_id`) instead of raw SILC codes (`2`, `5`, `BE33`,
`HB030`).


In [ ]:
# Human-readable label maps — defined once, reused throughout the notebook.
SEX = {1: "male", 2: "female"}                                    # RB090 / PB150

ACTIVITY_STATUS = {                                               # RB210 — basic activity status
    1: "at work", 2: "unemployed", 3: "retired", 4: "other inactive",
}

EDUCATION = {                                                     # PE040 — highest ISCED level
    1: "≤primary", 2: "lower sec", 3: "upper sec",
    4: "post-sec", 5: "tertiary", 6: "tertiary+",
}

TENURE = {                                                        # HH021 — housing tenure status
    1: "owner", 2: "owner (mortgage)", 3: "tenant (market)",
    4: "tenant (reduced)", 5: "free",
}

BE_REGIONS = {                                                    # DB040 — Belgian NUTS-2 regions
    "BE10": "Brussels",        "BE21": "Antwerp",
    "BE22": "Limburg",         "BE23": "East Flanders",
    "BE24": "Flemish Brabant", "BE25": "West Flanders",
    "BE31": "Walloon Brabant", "BE32": "Hainaut",
    "BE33": "Liège",           "BE34": "Luxembourg (BE)",
    "BE35": "Namur",
}

# Friendly names for the terse SILC column codes (used for display only).
COLUMN_LABELS = {
    "HB030": "household_id", "RB030": "person_id",
    "DB040": "region",       "DB090": "hh_weight",
    "HX050": "equiv_size",   "HX040": "hh_size",
}

print("Label maps ready:",
      ", ".join(f"{n} ({len(d)})" for n, d in [
          ("SEX", SEX), ("ACTIVITY_STATUS", ACTIVITY_STATUS),
          ("EDUCATION", EDUCATION), ("TENURE", TENURE),
          ("BE_REGIONS", BE_REGIONS), ("COLUMN_LABELS", COLUMN_LABELS)]))


---
## 1 · Advanced pandas

### 1.1 Memory Profiling — Why It Matters

Before optimising, we **measure**. Python's `psutil` gives us process-level RAM; pandas gives us DataFrame-level memory.

In [ ]:
import os
import psutil


def ram_mb() -> float:
    return psutil.Process(os.getpid()).memory_info().rss / 1024**2

# Baseline
ram_before = ram_mb()

# Load all Luxembourg years naively
lu_files = sorted((DATA_DIR / "LU_PUF_EUSILC").glob("LU_*h_EUSILC.csv"))
frames = [pd.read_csv(f) for f in lu_files]
df_lu_naive = pd.concat(frames, ignore_index=True)

ram_after = ram_mb()

print("Luxembourg H-files — naive load:")
print(f"  Shape  : {df_lu_naive.shape}")
print(f"  Memory : {df_lu_naive.memory_usage(deep=True).sum() / 1024**2:.1f} MB")
print(f"  RAM increase: {ram_after - ram_before:.1f} MB")
print()
print("dtypes:")
print(df_lu_naive.dtypes.value_counts())

### 1.2 Optimised dtypes + `usecols` — Memory Reduction

Two levers together give the real saving:

1. **`usecols`** — don't load columns you don't need. The H-file has 189 columns; we only use 8. Skipping 181 columns already cuts memory by ~95% before any dtype work.
2. **Optimised dtypes** — for the columns we *do* load:
   - Country codes (`"LU"`) → `category` (26 distinct values, not thousands of strings)
   - Year (2004–2023) → `int16` (not `int64`)
   - Incomes → `float32` (sub-euro precision not needed; saves half the float memory)
   - Household size → `int8` (values 1–15 fit in a single byte)

In [ ]:
# Define optimised dtype map for H-file columns
DTYPE_H = {
    "HB010": "int16",         # survey year
    "HB020": "category",       # country code
    "HB030": "int32",          # household ID
    "HB050": "int8",           # interview mode
    "HY010": "float32",        # gross total income
    "HY020": "float32",        # net disposable income
    "HX040": "int8",           # household size
    "HX050": "float32",        # equivalised size
    "DB090": "float32",        # design weight (from D merge)
}


def load_silc_h_optimised(
    country: str,
    years: list[int],
    data_dir: Path,
    dtype_map: dict | None = None,
) -> pd.DataFrame:
    """Load H-files for multiple years with optimised dtypes and column pruning."""
    dtype_map = dtype_map or DTYPE_H
    # Only load the columns we actually need — the real memory lever
    cols_to_load = set(k for k in dtype_map if k != "DB090")
    frames = []
    for year in years:
        path = data_dir / f"{country}_PUF_EUSILC" / f"{country}_{year}h_EUSILC.csv"
        if not path.exists():
            continue
        df = pd.read_csv(
            path,
            usecols=lambda c: c in cols_to_load,
            dtype={k: v for k, v in dtype_map.items() if k != "DB090"},
            na_values=["NA", ""],
            keep_default_na=True,
        )
        frames.append(df.assign(survey_year=np.int16(year)))
    return pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()


LU_YEARS = list(range(2004, 2014))
df_lu = load_silc_h_optimised("LU", LU_YEARS, DATA_DIR)

naive_mb  = df_lu_naive.memory_usage(deep=True).sum() / 1024**2
optim_mb  = df_lu.memory_usage(deep=True).sum() / 1024**2

print("Luxembourg 2004–2013 H-files:")
print(f"  Naive    memory: {naive_mb:.1f} MB  ({df_lu_naive.shape[1]} columns)")
print(f"  Optimised memory: {optim_mb:.1f} MB  ({df_lu.shape[1]} columns)")
print(f"  Reduction: {(1 - optim_mb/naive_mb)*100:.0f}%")
print(f"  Shape: {df_lu.shape}")
print()
print(df_lu[["HB020", "survey_year", "HY020", "HX040"]].dtypes)


### 1.3 Chunked Loading for Very Large Files

For files that exceed available RAM, `pd.read_csv(chunksize=...)` yields one chunk at a time. We process each chunk and discard it before reading the next.

In [ ]:
def compute_arop_chunked(
    path: Path,
    chunksize: int = 5000,
) -> dict:
    """
    Compute AROP in a single chunked pass — O(1) RAM regardless of file size.
    Strategy: two passes. First pass finds the median; second counts at-risk.
    Here we use a reservoir approach: accumulate incomes but only keep positives.
    For production, use T-Digest or a sorted file approach.
    """
    incomes = []
    for chunk in pd.read_csv(
        path,
        usecols=["HY020", "HX040"],
        dtype={"HY020": "float32", "HX040": "float32"},
        na_values=["NA", ""],
        chunksize=chunksize,
    ):
        print(f"Processing chunk with {len(chunk)} rows...")
        # Compute equivalised income for this chunk
        chunk["HX040"] = chunk["HX040"].fillna(1).clip(lower=1)
        chunk["equiv"] = chunk["HY020"] / (1.0 + 0.5 * (chunk["HX040"] - 1))
        # Keep only positive incomes (negatives excluded from threshold calc)
        pos = chunk.loc[chunk["equiv"] > 0, "equiv"].tolist()
        incomes.extend(pos)

    if not incomes:
        return {"n": 0, "median": 0.0, "threshold": 0.0, "arop": 0.0}

    incomes_sorted = sorted(incomes)
    median         = incomes_sorted[len(incomes_sorted) // 2]
    threshold      = 0.6 * median
    at_risk        = sum(1 for x in incomes if x < threshold)

    return {
        "n":         len(incomes),
        "median":    round(median, 2),
        "threshold": round(threshold, 2),
        "arop":      round(at_risk / len(incomes), 4),
    }


for country, year in [("ES", 2012), ("FR", 2012), ("BE", 2012)]:
    path = DATA_DIR / f"{country}_PUF_EUSILC" / f"{country}_{year}h_EUSILC.csv"
    if path.exists():
        result = compute_arop_chunked(path)
        print(f"{country} {year}: n={result['n']:,}  "
              f"median=€{result['median']:,.0f}  "
              f"AROP={result['arop']:.1%}")

### 1.4 Multi-Index DataFrames

With SILC data spanning (country × year × region), a **MultiIndex** makes hierarchical slicing natural:

```python
df.loc[("LU", 2012)]          # all Luxembourg 2012 rows
df.loc[("LU", slice(None))]   # all Luxembourg years
df.xs(2012, level="year")     # all countries for 2012
```

In [ ]:
# Build a multi-country, multi-year summary DataFrame
PARTICIPANT_COUNTRIES = ["BE", "ES", "FR", "EL", "HU", "IE", "LU", "SE"]
YEARS = list(range(2008, 2014))

from silc_toolkit import load_incomes, arop_rate, gini_coefficient

records = []
for country in PARTICIPANT_COUNTRIES:
    for year in YEARS:
        try:
            inc = load_incomes(country, year, DATA_DIR)
            if inc:
                records.append({
                    "country": country,
                    "year":    year,
                    "n_hh":   len(inc),
                    "median":  round(sorted(inc)[len(inc)//2], 0),
                    "arop":    round(arop_rate(inc), 4),
                    "gini":    round(gini_coefficient(inc), 4),
                })
        except FileNotFoundError:
            pass

# Create MultiIndex on (country, year)
df_summary = pd.DataFrame(records)
df_summary = df_summary.set_index(["country", "year"]).sort_index()

print("Multi-index summary (first 12 rows):")
df_summary.head(12)

In [ ]:
# MultiIndex slicing
print("=== Luxembourg AROP trend (2008–2013) ===")
print(df_summary.loc["LU", ["arop", "gini"]].to_string())

print("\n=== All countries, year 2012 ===")
print(df_summary.xs(2012, level="year")[["arop", "gini"]].sort_values("arop").to_string())

print("\n=== Countries with AROP > 25% in 2012 ===")
mask_2012 = df_summary.xs(2012, level="year")
high_risk  = mask_2012[mask_2012["arop"] > 0.25]
print(high_risk[["arop", "gini"]].to_string())

### 1.5 groupby, transform, and Rolling Windows

`groupby` is the pandas implementation of the **Split → Apply → Combine** pattern. `transform` is like `apply` but returns a Series with the same index as the original — perfect for adding back group-level statistics.

In [ ]:
# Reset index so 'country' and 'year' become regular columns
df = df_summary.reset_index()

# --- groupby: mean AROP per country ---
print("Mean AROP 2008–2013 by country:")
arop_by_country = (
    df.groupby("country")["arop"]
    .agg(["mean", "min", "max"])
    .round(4)
    .sort_values("mean")
)
print(arop_by_country.to_string())

# --- transform: add country-level median as a column ---
df["country_median_arop"] = df.groupby("country")["arop"].transform("median")
df["arop_vs_country_avg"]  = df["arop"] - df["country_median_arop"]

print("\nLuxembourg AROP deviation from country median:")
lu = df[df["country"] == "LU"][["year", "arop", "country_median_arop", "arop_vs_country_avg"]]
print(lu.to_string(index=False))

In [ ]:
# Rolling window: 3-year rolling average AROP per country
df_sorted = df.sort_values(["country", "year"])

df_sorted["arop_rolling3"] = (
    df_sorted
    .groupby("country")["arop"]
    .transform(lambda s: s.rolling(3, min_periods=1).mean())
)

print("3-year rolling AROP — selected countries:")
for c in ["IE", "HU", "LU"]:
    sub = df_sorted[df_sorted["country"] == c][["year", "arop", "arop_rolling3"]]
    print(f"\n  {c}:")
    print(sub.to_string(index=False))

### 1.6 — Merging & joining the four SILC files 🔗

So far we have worked one file at a time. But EU-SILC ships as a **star schema** of four
linked files per country-year, and the most interesting questions need *several of them at
once*. This is where `pandas.merge` earns its keep.

```
 D ── household register ─┐                       R ── personal register (everyone) ─┐
 H ── household data ─────┘  DB030 = HB030        P ── personal data (16+ only) ─────┘ RB030 = PB030
                                                  
        person → household:  RX030 = HB030   (household ID = personal ID // 10)
```

| Join | Keys | Cardinality | What it adds |
|---|---|---|---|
| **D ↔ H** | `DB030 = HB030` | one-to-one | region (`DB040`) + design weight (`DB090`) onto household income |
| **person → household** | `RX030 = HB030` | many-to-one | household income/region onto every individual |
| **R ↔ P** | `RB030 = PB030` | one-to-one* | adult labour/education onto the register (*children have no P row) |

Two `merge` arguments make joins **safe and self-documenting**, and you should use them
habitually:

- **`validate=`** (`"one_to_one"`, `"many_to_one"`, …) — pandas *raises* if the real
  cardinality differs, catching duplicate-key bugs that would silently fan out your rows.
- **`indicator=True`** — adds a `_merge` column (`both` / `left_only` / `right_only`) so you
  can *see* which rows matched. Here it neatly reveals the children who exist in `R` but not `P`.

In [ ]:
# Load the four BE 2012 files, keeping only the columns we need
YEAR   = 2012
country = "BE"
lu_dir = DATA_DIR / f"{country}_PUF_EUSILC"

df_d = pd.read_csv(lu_dir / f"{country}_{YEAR}d_EUSILC.csv",
                   usecols=["DB030", "DB040", "DB090"])                       # household register
df_h = pd.read_csv(lu_dir / f"{country}_{YEAR}h_EUSILC.csv",
                   usecols=["HB030", "HY020", "HX050", "HH021"])              # household data
df_r = pd.read_csv(lu_dir / f"{country}_{YEAR}r_EUSILC.csv",
                   usecols=["RB030", "RB080", "RB090", "RB210", "RX010", "RX030"])  # personal register
df_p = pd.read_csv(lu_dir / f"{country}_{YEAR}p_EUSILC.csv",
                   usecols=["PB030", "PE040", "PL031", "PY010G"])             # personal data (16+)

print(f"D households : {len(df_d):>6,}")
print(f"H households : {len(df_h):>6,}")
print(f"R persons    : {len(df_r):>6,}")
print(f"P persons 16+: {len(df_p):>6,}")

# --- D ↔ H: two household files, row-for-row (ONE-to-ONE) ------------------
hh = df_h.merge(
    df_d,
    left_on="HB030", right_on="DB030",
    how="inner",
    validate="one_to_one",         # pandas raises if the key is not unique on BOTH sides
    indicator=True,                # adds a _merge column showing the match source
)
hh["equiv_income"] = hh["HY020"] / hh["HX050"]   # equivalised disposable income
print(f"\nMerged household table: {hh.shape}")
print("merge indicator:", hh["_merge"].value_counts().to_dict())
hh = hh.drop(columns="_merge")

# readable view: decode the region code and give the SILC columns plain names
(hh[["HB030", "DB040", "DB090", "HX050", "equiv_income"]]
    .assign(DB040=lambda d: d["DB040"].map(BE_REGIONS).fillna(d["DB040"]))
    .rename(columns=COLUMN_LABELS)
    .sample(5))


In [ ]:
# --- person → household: MANY persons map to ONE household -----------------
# Linking rule (data dictionary): household ID = personal ID // 10, stored as RX030.
persons = df_r.merge(
    hh[["HB030", "DB040", "HH021", "equiv_income"]],
    left_on="RX030", right_on="HB030",
    how="left",
    validate="many_to_one",        # many persons ↔ one household
)

# --- R ↔ P: register (everyone) ↔ personal data (16+ only) -----------------
# A left join keeps children, who have no P-record; `indicator` exposes them.
persons = persons.merge(
    df_p,
    left_on="RB030", right_on="PB030",
    how="left",
    indicator=True,
)
print("R↔P merge outcome (left_only = children under 16):")
print(persons["_merge"].value_counts().to_string())

# --- decode numeric SILC codes to readable labels (maps defined at the top) -
persons["sex"]       = persons["RB090"].map(SEX)
persons["age"]       = persons["RX010"]
persons["activity"]  = persons["RB210"].map(ACTIVITY_STATUS)
persons["education"] = persons["PE040"].map(EDUCATION)
persons["tenure"]    = persons["HH021"].map(TENURE)
print(f"\nAnalytical person table: {persons.shape[0]:,} rows × {persons.shape[1]} cols")
(persons[["RB030", "age", "sex", "activity", "education", "tenure", "DB040", "equiv_income"]]
    .assign(DB040=lambda d: d["DB040"].map(BE_REGIONS).fillna(d["DB040"]))
    .rename(columns=COLUMN_LABELS)
    .sample(10))


### 1.7 — Reshaping: `pivot_table`, `crosstab`, `melt`

A merged table is rarely the final shape you want for analysis or plotting. The three core
reshaping verbs:

- **`pivot_table`** — spreadsheet-style cross-tabulation with an aggregation (mean, median, …).
- **`crosstab`** — frequency / proportion tables, with optional normalisation by row or column.
- **`melt`** — the inverse of pivoting: collapse many columns into *tidy* `(variable, value)` rows,
  the shape most plotting and modelling libraries expect.

We decode the numeric SILC codes to readable labels with `Series.map(...)`, then slice the
person table by **education**, **sex** and **economic activity**.

In [ ]:
# pivot_table — median equivalised income by education × sex (adults only)
adults = persons[persons["_merge"] == "both"]

pivot = pd.pivot_table(
    adults,
    values="equiv_income",
    index="education",
    columns="sex",
    #columns=["activity", "sex"],
    aggfunc="median",
    observed=True,
).round(0)

# present education levels in their natural order rather than alphabetically
edu_order = ["≤primary", "lower sec", "upper sec", "post-sec", "tertiary", "tertiary+"]
pivot = pivot.reindex([e for e in edu_order if e in pivot.index])
print("Median equivalised income by education × sex (€):")
pivot

In [ ]:
# crosstab — activity status composition within each sex
ct = pd.crosstab(persons["activity"], persons["sex"], normalize="columns")
print("\nActivity-status distribution within each sex (%):")
print((ct * 100).round(1).to_string())

In [ ]:
# melt — turn the wide education × sex table back into long ("tidy") form
wide = pivot.reset_index()
long = wide.melt(
    id_vars="education",
    value_vars=[c for c in wide.columns if c != "education"],
    var_name="sex",
    value_name="median_income",
)
print("Wide → long (tidy) via melt:")
long


### 1.8 — Binning & ranking: `cut`, `qcut`, `rank`

Continuous money and age variables become far more informative once **discretised**.
Three workhorses:

| Function | What it does | Typical use |
|---|---|---|
| `pd.qcut(x, q)` | **equal-frequency** bins (quantiles) | income deciles / quintiles |
| `pd.cut(x, bins)` | **fixed-width / custom-edge** bins | age brackets, score bands |
| `Series.rank(pct=True)` | position of each value in the distribution | percentile rank, gap analysis |

Below we turn equivalised income into deciles to eyeball inequality (what share of total
income does each decile hold?), bin ages into life-stage bands, and rank every household.

In [ ]:
# --- qcut: split households into equivalised-income deciles ----------------
hh_valid = hh[hh["equiv_income"] > 0].copy()
hh_valid["decile"] = pd.qcut(hh_valid["equiv_income"], 10, labels=False) + 1

income_share = (
    hh_valid.groupby("decile")["equiv_income"].sum()
    / hh_valid["equiv_income"].sum() * 100
)
print("Share of total equivalised income held by each decile (%):")
print(income_share.round(1).to_string())
print(f"  → top decile holds {income_share.iloc[-1]:.1f}%  vs  "
      f"bottom decile {income_share.iloc[0]:.1f}%")

# --- cut: fixed age bands, median income per band --------------------------
persons_hh = persons.dropna(subset=["equiv_income"])
age_bands = pd.cut(
    persons_hh["age"],
    bins=[0, 17, 34, 49, 64, 120],
    labels=["0–17", "18–34", "35–49", "50–64", "65+"],
)
income_by_age = (
    persons_hh.groupby(age_bands, observed=True)["equiv_income"].median().round(0)
)
print("\nMedian equivalised income by age band (€):")
print(income_by_age.to_string())

# --- rank: position of each household in the national distribution ---------
hh_valid["income_pctile"] = hh_valid["equiv_income"].rank(pct=True)
print("\nLowest-income households and their percentile:")
print(hh_valid[["HB030", "equiv_income", "decile", "income_pctile"]]
      .sort_values("equiv_income").head(3).round(3).to_string(index=False))

### 1.9 — Survey weights: the official-statistics correction ⚖️

Everything above computed **unweighted** statistics — and for *official* statistics that is
simply wrong. EU-SILC is a complex sample: households are drawn with unequal probabilities, then
adjusted for non-response and calibrated to population totals. Each record therefore carries a
**design weight** telling you how many real units it represents:

| Weight | File | Use |
|---|---|---|
| `DB090` | D — household register | **household-level** estimates |
| `RB050` | R — personal register | person-level estimates (AROP, poverty) |
| `PB040` | P — personal data (16+) | estimates restricted to adults |

A plain `df["x"].median()` treats every sampled record equally and will not reproduce the
published figure. The correct estimators are **weighted**: the weighted mean
$\bar{x}_w = \frac{\sum_i w_i x_i}{\sum_i w_i}$, and — for the poverty line and AROP — a
**weighted median / quantile**.

We demonstrate with the household weight `DB090`. Ignoring it visibly shifts both the median income and the at-risk share. We implement a
weighted quantile from scratch and compare it against the naïve unweighted version.

In [ ]:
# DB090 — the household design weight — says how many households each record represents.
hw = hh[hh["equiv_income"] > 0].copy()


def weighted_quantile(values, sample_weights, q):
    """Quantile q (0–1) of `values`, accounting for survey `sample_weights`."""
    values = np.asarray(values, dtype=float)
    sw     = np.asarray(sample_weights, dtype=float)
    order  = np.argsort(values)
    values, sw = values[order], sw[order]
    cum = (np.cumsum(sw) - 0.5 * sw) / sw.sum()    # Hazen plotting positions
    return np.interp(q, cum, values)


income = hw["equiv_income"].to_numpy()
w      = hw["DB090"].to_numpy()
print(f"household weights range {w.min():.1f} … {w.max():.1f} "
      f"(mean {w.mean():.1f}) — far from uniform\n")

# median: unweighted sample statistic vs design-weighted population estimate
median_unw = np.median(income)
median_w   = weighted_quantile(income, w, 0.5)

# at-risk share: below 60 % of the (respective) median
arop_unw = np.mean(income < 0.6 * median_unw) * 100
arop_w   = (w * (income < 0.6 * median_w)).sum() / w.sum() * 100

print(f"{'':26}{'unweighted':>12}{'weighted':>12}")
print(f"{'median equiv income  €':26}{median_unw:>12,.0f}{median_w:>12,.0f}")
print(f"{'poverty threshold    €':26}{0.6 * median_unw:>12,.0f}{0.6 * median_w:>12,.0f}")
print(f"{'at-risk share        %':26}{arop_unw:>12.1f}{arop_w:>12.1f}")
print(f"\n{len(hw):,} sampled households represent {w.sum():,.0f} in the population.")
print("→ Ignoring DB090 biases every estimate — the design weight is part of the data,")
print("  not an optional extra. Always use it for representative results!")

### 1.10 — Visualising inequality: the Lorenz curve & Gini  *(bonus / self-study)*

> 💡 *Bonus section — useful but not core to the morning. Skim it live and work through it on your own afterwards.*

The decile shares above already hint at inequality; the **Lorenz curve** makes it visual. It
plots the cumulative share of *income* against the cumulative share of *population*, ranked
poorest-to-richest. The further the curve bows below the 45° line of perfect equality, the more
unequal the distribution. The **Gini coefficient** is twice the area between them
($0$ = everyone equal, $1$ = one person holds everything):

$$ G = 1 - 2\int_0^1 L(p)\,dp $$

Crucially we build the **weighted** Lorenz curve, using the same survey weights from §1.9 — an
unweighted curve would misrepresent the population.

In [ ]:
import matplotlib.pyplot as plt


def lorenz_curve(values, weights):
    """Return cumulative population & income shares for a (weighted) Lorenz curve."""
    values  = np.asarray(values, dtype=float)
    weights = np.asarray(weights, dtype=float)
    order   = np.argsort(values)
    values, weights = values[order], weights[order]
    cum_pop = np.cumsum(weights)
    cum_inc = np.cumsum(values * weights)
    cum_pop = np.insert(cum_pop / cum_pop[-1], 0, 0.0)
    cum_inc = np.insert(cum_inc / cum_inc[-1], 0, 0.0)
    return cum_pop, cum_inc


def gini(cum_pop, cum_inc):
    """Gini = 1 − 2 × area under the Lorenz curve (trapezoidal rule)."""
    area = np.sum((cum_pop[1:] - cum_pop[:-1]) * (cum_inc[1:] + cum_inc[:-1]) / 2)
    return 1.0 - 2.0 * area


# reuse the weighted household income & weights from the previous cell
x, y = lorenz_curve(income, w)
g = gini(x, y)

fig, ax = plt.subplots(figsize=(5.5, 5.5))
ax.plot([0, 1], [0, 1], "--", color="grey", lw=1, label="Perfect equality")
ax.plot(x, y, color="#1f77b4", lw=2, label=f"Lorenz curve (Gini = {g:.3f})")
ax.fill_between(x, y, x, color="#1f77b4", alpha=0.12)
ax.set_xlabel("Cumulative share of population")
ax.set_ylabel("Cumulative share of equivalised income")
ax.set_title(f"{country} {YEAR} — income concentration")
ax.set_aspect("equal")
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
ax.legend(loc="upper left")
fig.tight_layout()
plt.show()

print(f"Weighted Gini coefficient: {g:.3f}  "
      f"(0 = perfect equality, 1 = one person holds all income)")

### 1.11 — Performance: vectorisation vs `.apply()`  *(bonus / self-study)*

> 💡 *Bonus section — useful but not core to the morning. Skim it live and work through it on your own afterwards.*

The single most common pandas performance mistake is reaching for `.apply()` (or worse,
`.iterrows()`) when a **vectorised** expression exists. `apply(axis=1)` runs a *Python-level
loop* and builds a `Series` for every row; the vectorised equivalent pushes the work down into
compiled C / NumPy and runs on the whole column at once.

A useful mental hierarchy, fastest → slowest:

1. **NumPy ufuncs** on `.to_numpy()` arrays — pure C, no pandas overhead.
2. **Vectorised pandas** (`col_a / col_b`, `pd.cut`, `.str`, `.where`, `np.select`).
3. **`.apply()`** — a Python callback per element/row.
4. **`.iterrows()` / explicit `for` loops** — almost always avoidable.

We time the same two derivations both ways. The answers are identical — only the clock changes.
Watch the order-of-magnitude gap on `apply(axis=1)`.

In [ ]:
# A common temptation: a row-wise .apply() to derive a categorical column
def to_band(income):
    if income < 20_000:
        return "low"
    elif income < 40_000:
        return "middle"
    return "high"

bins   = [-np.inf, 20_000, 40_000, np.inf]
labels = ["low", "middle", "high"]

print("Deriving an income band  (6,031 rows):")
print("  .apply(to_band)  — Python callback per row")
%timeit -n 20 -r 3 hh["equiv_income"].apply(to_band)
print("  pd.cut(...)      — vectorised, C-level")
%timeit -n 20 -r 3 pd.cut(hh["equiv_income"], bins=bins, labels=labels, right=False)

print("\nDividing two columns (6,031 rows):")
print("  apply(axis=1)    — builds a Series per row")
%timeit -n 5 -r 3 hh.apply(lambda r: r["HY020"] / r["HX050"], axis=1)
print("  NumPy division   — single C loop")
%timeit -n 200 -r 3 hh["HY020"].to_numpy() / hh["HX050"].to_numpy()

# Same answer, very different speed — always prefer the vectorised form
assert (
    pd.cut(hh["equiv_income"], bins=bins, labels=labels, right=False).astype(str)
    == hh["equiv_income"].apply(to_band)
).all()
print("\n✓ identical results — vectorising changes only the runtime, not the answer")

---
### 🏋️ Exercise 1 — Cross-Country Convergence

Using the multi-country `df_summary`:
1. Compute the **coefficient of variation** (std/mean) of AROP across participant countries for each year
2. Is AROP converging (CV decreasing) or diverging over 2008–2013?
3. Which country showed the **biggest single-year increase** in AROP?

**Hint:** `df.groupby("year")["arop"].agg(["std", "mean"])` gives you what you need.

In [ ]:
# 🏋️ Exercise 1 — Your solution here!


<details><summary>💡 Click for the solution</summary>

```python
# 1. Coefficient of variation per year
cv = (
    df.groupby("year")["arop"]
    .agg(std="std", mean="mean")
    .assign(cv=lambda x: x["std"] / x["mean"])
)
print("Coefficient of Variation of AROP across participant countries:")
print(cv.round(4).to_string())

# 2. Trend
if cv["cv"].iloc[-1] < cv["cv"].iloc[0]:
    print("\nTrend: CONVERGING (CV decreasing)")
else:
    print("\nTrend: DIVERGING (CV increasing)")

# 3. Biggest single-year jump
df_s = df.sort_values(["country", "year"])
df_s["arop_change"] = df_s.groupby("country")["arop"].diff()
biggest = df_s.loc[df_s["arop_change"].idxmax()]
print(f"\nBiggest single-year AROP jump: {biggest['country']} in {int(biggest['year'])} "
      f"(+{biggest['arop_change']:.1%})")
```
</details>

---
### 🏋️ Exercise 2 — Put the joins & reshapes together

You now have a person-level table (`persons`) carrying household income, region and
demographics, plus a household-level table (`hh`). Use them to answer three questions:

1. **Tenure × education** — a `pivot_table` of median equivalised income.
2. **Gender income gap** — median male minus median female income, *per* education level.
3. **Quintile concentration** — `qcut` adults into 5 income quintiles, then `crosstab`
   quintile × education (row-normalised) to see where each education level sits.

The solution is hidden below — try it before peeking.

In [ ]:
# Your turn — work from the `persons` / `hh` tables built above.
#
# 1. pivot_table: median `equiv_income` by tenure status (HH021) × education.
# 2. Gender income gap: median male − median female income, per education level.
# 3. qcut adults into 5 income quintiles, then crosstab quintile × education
#    (normalised by row) to see how education concentrates across the distribution.

# ... your code here ...


In [ ]:
# 1. Median equivalised income by tenure × education
#    `persons` already carries a decoded "tenure" column (via the TENURE map above);
#    here we just keep the adult rows.
adults = persons[persons["_merge"] == "both"].copy()

piv = pd.pivot_table(
    adults, values="equiv_income",
    index="tenure", columns="education", aggfunc="median", observed=True,
).round(0)
print("Median equivalised income by tenure × education (€):")
print(piv.to_string())

# 2. Gender income gap per education level
gap = (
    adults.groupby(["education", "sex"], observed=True)["equiv_income"]
    .median()
    .unstack("sex")
    .assign(gap=lambda d: d["male"] - d["female"])
)
print("\nMale − female median income gap by education (€):")
print(gap.round(0).to_string())

# 3. Quintile × education concentration
adults["quintile"] = pd.qcut(adults["equiv_income"], 5, labels=range(1, 6))
conc = pd.crosstab(
    adults["quintile"],
    adults["education"],
    normalize="index"
    )
print("\nEducation composition within each income quintile (%):")
print((conc[EDUCATION.values()] * 100).round(1).to_string())

<details><summary>💡 Click for the solution</summary>

```python
# 1. Median equivalised income by tenure × education
#    `persons` already carries a decoded "tenure" column (via the TENURE map above);
#    here we just keep the adult rows.
adults = persons[persons["_merge"] == "both"].copy()

piv = pd.pivot_table(
    adults, values="equiv_income",
    index="tenure", columns="education", aggfunc="median", observed=True,
).round(0)
print("Median equivalised income by tenure × education (€):")
print(piv.to_string())

# 2. Gender income gap per education level
gap = (
    adults.groupby(["education", "sex"], observed=True)["equiv_income"]
    .median()
    .unstack("sex")
    .assign(gap=lambda d: d["male"] - d["female"])
)
print("\nMale − female median income gap by education (€):")
print(gap.round(0).to_string())

# 3. Quintile × education concentration
adults["quintile"] = pd.qcut(adults["equiv_income"], 5, labels=range(1, 6))
conc = pd.crosstab(adults["quintile"], adults["education"], normalize="index")
print("\nEducation composition within each income quintile (%):")
print((conc * 100).round(1).to_string())
```
</details>


---
## 2 · Polars — Modern High-Performance DataFrames

### 2.0 Why Polars?

| Feature | pandas | Polars |
|---------|--------|--------|
| Execution | Eager (immediate) | **Lazy by default** (optimised query plan) |
| Memory | Single-threaded | **Multi-threaded**, Apache Arrow columnar |
| Speed | Baseline | **5–50× faster on large aggregations** (but *slower* on tiny ones — overhead) |
| API | Mutable DataFrames | **Immutable** — fewer surprises |
| Missing values | `NaN` / `None` mixed | **`null` everywhere** — consistent |
| Syntax | `.groupby().agg()` | `.group_by().agg()` — similar but cleaner |

Polars is not a drop-in replacement for pandas, but for **aggregation-heavy SILC workflows** it shines.

In [ ]:
import polars as pl
print(f"Polars {pl.__version__}")

# Load Luxembourg 2012 H-file into Polars.
# pl.read_csv() is the eager reader: it loads the whole file into memory immediately
# (the lazy counterpart, pl.scan_csv(), is shown further down).
lu_h_path = DATA_DIR / "LU_PUF_EUSILC" / "LU_2012h_EUSILC.csv"

df_pl = pl.read_csv(
    lu_h_path,
    # infer_schema_length: how many rows Polars samples to guess each column's type.
    # A large value makes type inference robust on columns that are empty near the top.
    infer_schema_length=10000,
    # null_values: strings that should be read as `null` rather than literal text.
    # SILC uses "NA" and blanks for missing data.
    null_values=["NA", ""],
    # schema_overrides: pin specific columns to an explicit dtype instead of letting
    # Polars infer them — the equivalent of pandas' `dtype=` argument. Smaller types
    # (Int8/Int16/Float32) save memory; Categorical stores repeated strings once.
    schema_overrides={
        "HB010": pl.Int16,        # survey year
        "HB020": pl.Categorical,  # country code (one repeated value → store once)
        "HB030": pl.Int32,        # household ID
        "HY020": pl.Float32,      # net disposable income
        "HX040": pl.Int8,         # household size (values 1–15 fit in one byte)
    },
)

print(f"Shape: {df_pl.shape}")
# .select(...) picks a subset of columns (like pandas df[[...]]);
# .sample(10) draws 10 random rows for a quick eyeball.
print(df_pl.select(["HB020", "HB030", "HY020", "HX040"]).sample(10))


In [ ]:
# Compute AROP in Polars — three steps:
#   1. derive equivalised income, 2. find median → poverty threshold, 3. flag & average.

# --- Step 1: add an `equiv_income` column ---------------------------------
# .with_columns() returns a NEW DataFrame with extra/!replaced columns (Polars is
# immutable — operations never mutate in place, unlike pandas assignment).
# Inside it we build an *expression*: a recipe Polars applies to whole columns at once.
#   pl.col("X")        → refer to column X
#   pl.when().then().otherwise()  → vectorised if/else (no Python loop)
#   .cast(pl.Float32)  → change a column's dtype mid-expression
#   .alias("name")     → name the resulting column
df_pl = df_pl.with_columns([
    pl.when(pl.col("HX040").is_null())           # if household size is missing…
      .then(pl.col("HY020"))                     # …fall back to the raw income
      .otherwise(                                # …otherwise divide by the OECD scale
          pl.col("HY020") / (1.0 + 0.5 * (pl.col("HX040").cast(pl.Float32) - 1.0))
      )
      .alias("equiv_income")
])

# --- Step 2: keep only positive incomes -----------------------------------
# .filter() keeps the rows where the boolean expression is True (like pandas masking).
df_pos = df_pl.filter(pl.col("equiv_income") > 0)

# Pull single scalar values out of a column. Indexing a Polars column with ["name"]
# gives a Series; .median() reduces it to one number.
median_val  = df_pos["equiv_income"].median()
threshold   = 0.6 * median_val

# --- Step 3: flag at-risk households and take the share -------------------
arop_polars = (
    df_pos
    # add a boolean "at_risk" column (True when below the threshold)
    .with_columns((pl.col("equiv_income") < threshold).alias("at_risk"))
    # the mean of a boolean column = the proportion of True values = the AROP rate
    .select(pl.col("at_risk").mean().alias("arop"))
    # .item() extracts the single value from a 1×1 DataFrame into a plain Python float
    .item()
)

print(f"Luxembourg 2012 AROP (Polars):")
print(f"  Median income : €{median_val:,.0f}")
print(f"  Threshold     : €{threshold:,.0f}")
print(f"  AROP rate     : {arop_polars:.1%}")


In [ ]:
# Polars: multi-year AROP for Luxembourg — the LAZY API.
# A LazyFrame is NOT data — it's a *query plan*. Nothing is read or computed until you
# call .collect(). This lets Polars optimise the whole pipeline (e.g. only read the
# columns you actually use) before touching a single byte on disk.

lu_files = sorted((DATA_DIR / "LU_PUF_EUSILC").glob("LU_*h_EUSILC.csv"))

# Build one lazy query per yearly file. pl.scan_csv() is lazy (pl.read_csv() is eager).
lazy_frames = [
    pl.scan_csv(
        f,
        null_values=["NA", ""],
        schema_overrides={"HY020": pl.Float32, "HX040": pl.Float32},
        infer_schema_length=10000,
    )
      # .select() here keeps only the two columns we need, so every file shares the
      # SAME schema — a requirement for stacking them together below.
      .select(["HY020", "HX040"])
      # add a "year" column parsed from the filename (e.g. "LU_2012h..." → 2012).
      # pl.lit() turns a Python value into a constant column expression.
      .with_columns(pl.lit(int(f.name[3:7])).cast(pl.Int16).alias("year"))
    for f in lu_files
]

# pl.concat() stacks the per-year query plans vertically — still lazy, no data loaded yet.
df_lazy = pl.concat(lazy_frames)

# Append the equivalised-income expression, then filter — still just building the plan.
df_lazy = df_lazy.with_columns(
    (pl.col("HY020") / (
        # fill_null(1.0): replace missing household size with 1
        # .clip(lower_bound=1.0): never let the size drop below 1
        1.0 + 0.5 * (pl.col("HX040").fill_null(1.0).clip(lower_bound=1.0) - 1.0)
    )).alias("equiv")
).filter(pl.col("equiv") > 0)

# .collect() executes the ENTIRE plan in one optimised, multi-threaded pass.
df_collected = df_lazy.collect()
print(f"Collected: {df_collected.shape[0]:,} rows across {len(lu_files)} years")

# Group by year and compute AROP
def polars_arop_by_year(df: pl.DataFrame) -> pl.DataFrame:
    """Compute AROP per year using Polars group_by."""
    # 1) median income within each year → one row per year
    medians = df.group_by("year").agg(pl.col("equiv").median().alias("median"))
    # 2) join the per-year median back onto every row (like pandas merge/transform)
    df2 = df.join(medians, on="year")
    return (
        df2.with_columns(
            # flag rows below 60% of THEIR year's median
            (pl.col("equiv") < 0.6 * pl.col("median")).alias("at_risk")
        )
        # mean of the boolean flag = AROP share, computed per year
        .group_by("year")
        .agg(pl.col("at_risk").mean().alias("arop"))
        .sort("year")   # group_by output is unordered → sort for a tidy table
    )

arop_by_year = polars_arop_by_year(df_collected)
print("\nLuxembourg AROP by year (Polars):")
print(arop_by_year.with_columns(pl.col("arop").round(4)))


In [ ]:
# Benchmark: pandas vs Polars — poverty share per (country, year) across the WHOLE PUF.
# (We use net household income HY020 directly — the one income column present in *every*
#  country's H-file — so the job stays comparable across all 26 PUFs.)
#
# Each H-file *is* exactly one country-year, so the "<60% of the national median" share is
# computable file-by-file, independently. That makes the job embarrassingly parallel:
#   • pandas  → a serial, GIL-bound Python loop over files. (Global Interpreter Lock)
#   • Polars  → pl.collect_all(...) runs all per-file queries across every core at once.
import time

# Every household file across every country and year shipped in the PUF.
ALL_FILES = sorted(
    f
    for d in sorted(DATA_DIR.glob("*_PUF_EUSILC"))
    for f in d.glob("*_*h_EUSILC.csv")
)
n_files     = len(ALL_FILES)
n_countries = len({f.name[:2] for f in ALL_FILES})


def arop_pandas() -> pd.DataFrame:
    """Idiomatic pandas: a serial loop of read_csv + per-file poverty share."""
    rows = []
    for f in ALL_FILES:
        s = pd.read_csv(
            f,
            usecols=["HY020"],
            dtype={"HY020": "float32"},
            na_values=["NA", ""])["HY020"]
        s = s[s > 0]
        rows.append((f.name[:2], int(f.name[3:7]), float((s < 0.6 * s.median()).mean())))
    return pd.DataFrame(rows, columns=["country", "year", "arop"])


def _poverty_share(f):
    """Build a LAZY per-file query (a plan, not a result).

    Because it's a LazyFrame, Polars can later run many of these in parallel and
    optimise each one (e.g. read only HY020 from disk — "projection push-down").
    """
    return (
        pl.scan_csv(
            f,
            null_values=["NA", ""],                        # lazy reader
            schema_overrides={"HY020": pl.Float32},
            infer_schema_length=100
            )
          .select("HY020")                                 # only need this one column
          .filter(pl.col("HY020") > 0)                     # drop non-positive incomes
          .select(
              # tag each result with its country & year (constants via pl.lit)
              pl.lit(f.name[:2]).alias("country"),
              pl.lit(int(f.name[3:7])).cast(pl.Int16).alias("year"),
              # at-risk share in a single pass: mean of the boolean "below threshold"
              (pl.col("HY020") < 0.6 * pl.col("HY020").median()).mean().alias("arop"),
          )
    )


def arop_polars() -> "pl.DataFrame":
    """Idiomatic Polars: collect ALL per-file queries in parallel across the thread pool."""
    # pl.collect_all() executes a LIST of LazyFrames together, spreading them over every
    # CPU core. pl.concat() then stacks the per-file results into one DataFrame.
    return pl.concat(pl.collect_all([_poverty_share(f) for f in ALL_FILES]))


def best_ms(fn, runs: int = 3) -> float:
    """Warm up once (thread pool, query optimiser, OS file cache), then take the best of N."""
    fn()  # warm-up — pays the one-time costs we do NOT want to measure
    return min(
        (lambda t0=time.perf_counter(): (fn(), time.perf_counter() - t0)[1])()
        for _ in range(runs)
    ) * 1000

# Count total households the same lazy-parallel way: one pl.len() query per file.
n_rows  = sum(df.item() for df in pl.collect_all(
    [pl.scan_csv(f, null_values=["NA", ""], infer_schema_length=100).select(pl.len())
     for f in ALL_FILES]
))
time_pd = best_ms(arop_pandas)
time_pl = best_ms(arop_polars)

print(f"Benchmark — poverty share per (country, year) over the FULL PUF")
print(f"  {n_files} files · {n_countries} countries · {n_rows:,} households · best of 3 warm runs")
print(f"  pandas : {time_pd:7.1f} ms")
print(f"  Polars : {time_pl:7.1f} ms")
winner = "Polars" if time_pl < time_pd else "pandas"
print(f"  → {winner} wins ({max(time_pd, time_pl) / min(time_pd, time_pl):.1f}× faster).")
print()
print("Same algorithm, very different engines: pandas reads every CSV in a serial,")
print("GIL-bound Python loop. Polars' pl.collect_all dispatches all 187 per-file queries")
print("across every core at once, each one a fused scan→filter→aggregate plan. At this")
print("scale the parallelism dominates the fixed start-up cost — exactly where Polars wins.")


> **The honest takeaway for official statistics:** *pick the tool to fit the workload.* pandas is
> perfectly fast — often faster — for a single country-year; reach for Polars when the data
> outgrows a single file and the work becomes I/O- and aggregation-heavy. Same answers either
> way; only the clock changes.

---
## 3 · Parquet — Efficient Storage

Parquet is a **columnar binary format** that:
- Compresses 10–20× better than CSV
- Reads 5–50× faster (columnar access = only load columns you need)
- Preserves dtypes exactly (no string→int guessing)
- Is the standard in modern data pipelines (Spark, DuckDB, BigQuery)

We'll save our multi-country summary to Parquet and benchmark read times.

In [ ]:
import os

# Build a multi-country, all-years Parquet dataset
from silc_toolkit.loaders import load_household_df

PARTICIPANT_COUNTRIES = ["BE", "ES", "FR", "EL", "HU", "IE", "LU", "SE"]
ALL_YEARS = list(range(2008, 2014))

parquet_dir = PROJECT_ROOT / "data" / "parquet"
parquet_dir.mkdir(exist_ok=True)

for country in PARTICIPANT_COUNTRIES:
    frames = []
    for year in ALL_YEARS:
        try:
            df = load_household_df(country, year, DATA_DIR)
            df["survey_year"] = year
            frames.append(df)
        except FileNotFoundError:
            pass
    if frames:
        df_country = pd.concat(frames, ignore_index=True)
        out_path = parquet_dir / f"{country}_households.parquet"
        df_country.to_parquet(out_path, index=False, compression="snappy")  # 'zstd' (smaller files size, slower compression)
        csv_size_mb = sum(
            os.path.getsize(DATA_DIR / f"{country}_PUF_EUSILC" / f"{country}_{y}h_EUSILC.csv")
            for y in ALL_YEARS
            if (DATA_DIR / f"{country}_PUF_EUSILC" / f"{country}_{y}h_EUSILC.csv").exists()
        ) / 1024**2
        pq_size_mb = os.path.getsize(out_path) / 1024**2
        print(f"  {country}: CSV={csv_size_mb:.1f}MB → Parquet={pq_size_mb:.1f}MB "
              f"({100*(1-pq_size_mb/csv_size_mb):.0f}% smaller)")

print("\n✅ Parquet files written to", parquet_dir)

In [ ]:
# Benchmark: CSV vs Parquet read time for Belgium
be_csv_files = sorted((DATA_DIR / "BE_PUF_EUSILC").glob("BE_*h_EUSILC.csv"))
be_parquet   = parquet_dir / "BE_households.parquet"

# CSV read
t0 = time.perf_counter()
df_csv = pd.concat([pd.read_csv(f) for f in be_csv_files if f.exists()])
time_csv = time.perf_counter() - t0

# Parquet read
t0 = time.perf_counter()
df_pq = pd.read_parquet(be_parquet)
time_pq = time.perf_counter() - t0

print(f"Belgium read benchmark:")
print(f"  CSV     : {time_csv:.3f}s  ({len(df_csv):,} rows)")
print(f"  Parquet : {time_pq:.3f}s  ({len(df_pq):,} rows)")
print(f"  Speedup : {time_csv/time_pq:.1f}×")

# Parquet column pruning — only load what you need
t0 = time.perf_counter()
df_light = pd.read_parquet(be_parquet, columns=["HB030", "HY020", "equiv_income"])
time_light = time.perf_counter() - t0
print(f"  Parquet (3 cols only): {time_light:.3f}s — column pruning!")

In [ ]:
%%writefile silc_toolkit/loaders.py
"""
CSV, pandas, and Parquet loaders for EU-SILC PUF data.

All functions accept a ``data_dir`` argument so they work with any copy
of the PUF, regardless of installation path.
"""
from __future__ import annotations

import csv
import logging
from pathlib import Path
from typing import Optional

import pandas as pd

logger = logging.getLogger(__name__)

H_COLS_MINIMAL = ["HB010", "HB020", "HB030", "HY010", "HY020", "HX040"]
D_COLS_MINIMAL = ["DB010", "DB020", "DB030", "DB040", "DB090"]

DTYPE_H = {
    "HB010": "int16",
    "HB020": "category",
    "HB030": "int32",
    "HY010": "float32",
    "HY020": "float32",
    "HX040": "float32",
}


def load_incomes(
    country: str,
    year: int,
    data_dir: Path,
    *,
    min_income: float = 0.0,
    max_rows: Optional[int] = None,
) -> list[float]:
    """Load equivalised disposable incomes from the H-file."""
    path = data_dir / f"{country}_PUF_EUSILC" / f"{country}_{year}h_EUSILC.csv"
    if not path.exists():
        raise FileNotFoundError(f"No H-file for {country} {year}: {path}")
    incomes: list[float] = []
    with open(path, newline="", encoding="utf-8") as fh:
        for i, row in enumerate(csv.DictReader(fh)):
            if max_rows is not None and i >= max_rows:
                break
            inc_str  = row.get("HY020", "").strip()
            size_str = row.get("HX040", "").strip()
            if not inc_str or inc_str == "NA":
                continue
            income = float(inc_str)
            size   = max(1, int(float(size_str))) if size_str and size_str != "NA" else 1
            equiv  = income / (1.0 + 0.5 * (size - 1))
            if equiv >= min_income:
                incomes.append(equiv)
    return incomes


def load_household_df(
    country: str,
    year: int,
    data_dir: Path,
    cols: Optional[list[str]] = None,
) -> pd.DataFrame:
    """Load H-file for one country-year; merge with D-file for weight+region."""
    cols = cols or H_COLS_MINIMAL
    h_path = data_dir / f"{country}_PUF_EUSILC" / f"{country}_{year}h_EUSILC.csv"
    d_path = data_dir / f"{country}_PUF_EUSILC" / f"{country}_{year}d_EUSILC.csv"
    if not h_path.exists():
        raise FileNotFoundError(f"H-file not found: {h_path}")
    df_h = pd.read_csv(h_path, dtype={k: v for k, v in DTYPE_H.items()},
                       na_values=["NA", ""])
    available_h = [c for c in cols if c in df_h.columns]
    df_h = df_h[available_h].copy()
    if d_path.exists():
        df_d = pd.read_csv(d_path, usecols=lambda c: c in D_COLS_MINIMAL)
        if "DB030" in df_d.columns and "HB030" in df_h.columns:
            df = df_h.merge(
                df_d[[c for c in ["DB030", "DB090", "DB040"] if c in df_d.columns]],
                left_on="HB030", right_on="DB030", how="left",
            ).drop(columns="DB030", errors="ignore")
        else:
            df = df_h
    else:
        df = df_h
    for col in ["HY010", "HY020"]:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce").fillna(0.0)
    if "HX040" in df.columns:
        df["HX040"] = pd.to_numeric(df["HX040"], errors="coerce").fillna(1).clip(lower=1)
    if "DB090" in df.columns:
        df["DB090"] = pd.to_numeric(df["DB090"], errors="coerce").fillna(1.0)
    if "HY020" in df.columns and "HX040" in df.columns:
        df["equiv_income"] = df["HY020"] / (1.0 + 0.5 * (df["HX040"] - 1)).clip(lower=1)
    return df


def load_multi_year(
    country: str,
    years: list[int],
    data_dir: Path,
) -> pd.DataFrame:
    """Concatenate household DataFrames across multiple years for one country."""
    frames = []
    for year in years:
        try:
            df = load_household_df(country, year, data_dir)
            df["survey_year"] = year
            frames.append(df)
        except FileNotFoundError:
            logger.warning("No H-file for %s %d — skipped", country, year)
    if not frames:
        raise ValueError(f"No data found for {country} in years {years}")
    return pd.concat(frames, ignore_index=True)


def load_parquet(
    country: str,
    parquet_dir: Path,
    columns: Optional[list[str]] = None,
) -> pd.DataFrame:
    """Load a pre-built Parquet file for one country."""
    path = parquet_dir / f"{country}_households.parquet"
    if not path.exists():
        raise FileNotFoundError(f"No Parquet for {country}: {path}")
    return pd.read_parquet(path, columns=columns)

---
### 🏋️ Exercise 3 — Your Country in Polars

Choose a country with PUF data (see the list above). Using Polars:
1. Load all available years for your country with `pl.scan_csv` (lazy)
2. Compute equivalised income
3. Group by year and compute **both** AROP rate and median income in one `.agg()`
4. Print the results sorted by year

**Bonus:** Also compute the S80/S20 ratio using Polars expressions:
```python
pl.col("equiv").quantile(0.8) / pl.col("equiv").quantile(0.2)
```

In [ ]:
# 🏋️ Exercise 3 — Your solution here!
# MY_COUNTRY = "BE"   # change to your country
# MY_FILES = sorted((DATA_DIR / f"{MY_COUNTRY}_PUF_EUSILC").glob(f"{MY_COUNTRY}_*h_EUSILC.csv"))


<details><summary>💡 Click for the solution (Belgium example)</summary>

```python
MY_COUNTRY = "BE"
MY_FILES = sorted((DATA_DIR / f"{MY_COUNTRY}_PUF_EUSILC").glob(f"{MY_COUNTRY}_*h_EUSILC.csv"))

lazy = pl.concat([
    pl.scan_csv(f, null_values=["NA", ""],
                schema_overrides={"HY020": pl.Float32, "HX040": pl.Float32})
      .select(["HY020", "HX040"])
      .with_columns(pl.lit(int(f.name[3:7])).alias("year"))
    for f in MY_FILES
])

result = (
    lazy
    .with_columns(
        (pl.col("HY020") /
         (1.0 + 0.5 * (pl.col("HX040").fill_null(1.0).clip(lower_bound=1.0) - 1.0))
        ).alias("equiv")
    )
    .filter(pl.col("equiv") > 0)
    .collect()
)

medians = result.group_by("year").agg(pl.col("equiv").median().alias("median"))
result2 = result.join(medians, on="year")

summary = (
    result2.with_columns(
        (pl.col("equiv") < 0.6 * pl.col("median")).alias("at_risk")
    )
    .group_by("year")
    .agg([
        pl.col("at_risk").mean().alias("arop"),
        pl.col("equiv").median().alias("median_income"),
        # Bonus: approximate S80/S20
        (pl.col("equiv").quantile(0.80) / pl.col("equiv").quantile(0.20)).alias("s80s20"),
    ])
    .sort("year")
)
print(f"{MY_COUNTRY} — AROP, median income, S80/S20 by year:")
print(summary.with_columns([pl.col("arop").round(4), pl.col("median_income").round(0),
                             pl.col("s80s20").round(2)]))
```
</details>

---
## 🗺️ Recap

| Concept | Key syntax | SILC application |
|---------|-----------|------------------|
| **Optimised dtypes** | `dtype={"HB020": "category"}`, `usecols` | RAM reduction on H-files |
| **Chunked reading** | `pd.read_csv(chunksize=N)` | AROP without loading full CSV |
| **MultiIndex** | `df.set_index(["country", "year"])` | Slice by country-year efficiently |
| **groupby + agg** | `.groupby("country")["arop"].agg(["mean", "std"])` | AROP convergence analysis |
| **transform** | `.groupby("country").transform("median")` | Add country median as column |
| **rolling** | `.rolling(3, min_periods=1).mean()` | 3-year smoothed AROP trend |
| **Merging** | `df_p.merge(df_h, validate=..., indicator=True)` | Join D/H/R/P files safely |
| **Reshaping** | `pivot_table` · `crosstab` · `melt` | Tenure × education tables |
| **Binning & ranking** | `cut` · `qcut` · `rank` | Income quintiles & age bands |
| **Survey weights** | `weighted_quantile(x, w=DB090)` | Population-correct median & AROP |
| **Lorenz & Gini** | weighted cumulative shares | Inequality of equivalised income |
| **Vectorisation** | NumPy/`Series` ops over `.apply()` | 10–100× faster row computations |
| **Polars lazy** | `pl.scan_csv().filter().collect()` | Query-optimised multi-year load |
| **Polars group_by** | `.group_by().agg([pl.col().mean()])` | AROP and Gini in one pass |
| **Parquet write** | `df.to_parquet(path, compression="snappy")` | 70% smaller than CSV |
| **Parquet read** | `pd.read_parquet(path, columns=[...])` | Column pruning — 5× faster |

---
## ⏭️ Up Next

**Notebook 04 — Web Scraping & SQLAlchemy** (this afternoon)

- Scrape the Eurostat SILC documentation page with BeautifulSoup
- Navigate pagination, extract tables, save to CSV
- Design a normalised SQLite schema for SILC data
- Load our Parquet files into SQLite via SQLAlchemy ORM
- Reproduce AROP with a SQL query

☕ *Lunch break — back at 13:30!*